In [ ]:
import sys
print("🔧 Instalando dependencias: pydantic, email-validator")
!{sys.executable} -m pip install -q pydantic email-validator
print("✅ Instalación completada. Ejecuta las celdas siguientes.")

# AEM4L1 E02 — Pydantic: Validación y Tipos Avanzados

**Objetivo:** Aprender a usar validadores, tipos especiales (constr, EmailStr) y listas en Pydantic.

**Tipo:** Resuelto

**Uso:** listo para Google Colab. Ejecutá las celdas en orden.

## 📦 Dependencias

**Necesario instalar:**
- `pydantic` — librería de validación de datos
- `email-validator` — validador de emails para Pydantic

**Cómo instalar:**
```bash
pip install pydantic email-validator
```

La celda de instalación está incluida a continuación. Ejecuta esa celda primero.

In [ ]:
import sys

print("=" * 60)
print("🔧 Instalando dependencias")
print("=" * 60)

# Pydantic es la librería principal
print("\n📦 Instalando: pydantic")
!{sys.executable} -m pip install -q pydantic

# email-validator es necesario para EmailStr
print("📦 Instalando: email-validator")
!{sys.executable} -m pip install -q email-validator

print("\n✅ ¡Listo! Todas las dependencias instaladas.\n")

## Importaciones necesarias

En este notebook vamos a usar:
- **pydantic**: `BaseModel`, `field_validator`, `ValidationError`, `EmailStr`, `constr`
- **typing**: `List`, `Optional`, `Annotated` (tipos de datos avanzados)
- **datetime**: `date` (para validar fechas)
- **email_validator**: Dependencia para validar emails

## 1. Validadores personalizados con @field_validator

In [ ]:
from pydantic import BaseModel, field_validator, ValidationError

print("=" * 60)
print("EJEMPLO 1: Validadores Personalizados con @field_validator")
print("=" * 60)

class Usuario(BaseModel):
    """
    Modelo de Usuario con VALIDADORES PERSONALIZADOS.
    
    Los validadores permiten:
    - Limpiar datos (trim, lowercase)
    - Validar rangos
    - Validar formatos personalizados
    - Transformar datos
    """
    nombre: str
    edad: int
    
    @field_validator('nombre')
    @classmethod
    def nombre_no_vacio(cls, v):
        """Valida que el nombre no esté vacío y elimina espacios."""
        if not v or len(v.strip()) == 0:
            raise ValueError('El nombre no puede estar vacío')
        return v.strip()  # Limpia espacios al inicio/final
    
    @field_validator('edad')
    @classmethod
    def edad_valida(cls, v):
        """Valida que la edad esté en rango válido."""
        if v < 0 or v > 150:
            raise ValueError('La edad debe estar entre 0 y 150 años')
        return v

print("\n✅ Caso 1: Datos VÁLIDOS (con espacios extra):")
try:
    usuario1 = Usuario(nombre="  Juan  ", edad=30)
    print(f"   Usuario creado: {usuario1}")
    print(f"   - Nombre (limpio): '{usuario1.nombre}'")
    print(f"   - Edad: {usuario1.edad}")
except ValidationError as e:
    print(f"   ❌ Error: {e}")

print("\n❌ Caso 2: Edad fuera de rango:")
try:
    usuario2 = Usuario(nombre="Maria", edad=200)
except ValidationError as e:
    print(f"   ❌ Error validación: {e.errors()[0]['msg']}")

print("\n❌ Caso 3: Nombre vacío:")
try:
    usuario3 = Usuario(nombre="   ", edad=25)
except ValidationError as e:
    print(f"   ❌ Error validación: {e.errors()[0]['msg']}")

## 2. Tipos especiales: EmailStr y constr

In [ ]:
from pydantic import EmailStr
from pydantic import constr

print("\n" + "=" * 60)
print("EJEMPLO 2: Tipos Especiales - EmailStr y constr")
print("=" * 60)

class CuentaUsuario(BaseModel):
    """
    Modelo de cuenta con tipos ESPECIALES de validación.
    
    - EmailStr: Valida automáticamente que sea un email válido
    - constr(min_length=8): String con restricción de longitud mínima
    """
    email: EmailStr                    # Valida formato email automático
    contraseña: constr(min_length=8)  # Mínimo 8 caracteres
    usuario: str

print("\n✅ Caso 1: Datos VÁLIDOS:")
try:
    cuenta1 = CuentaUsuario(
        email="juan@example.com",
        contraseña="miContraseña123",
        usuario="juan_perez"
    )
    print(f"   Cuenta creada: {cuenta1}")
except ValidationError as e:
    print(f"   ❌ Error: {e}")

print("\n❌ Caso 2: Email INVÁLIDO:")
try:
    cuenta2 = CuentaUsuario(
        email="no_es_email",  # Falta el @
        contraseña="miContraseña123",
        usuario="maria"
    )
except ValidationError as e:
    print(f"   ❌ Error: {e.errors()[0]['msg']}")

print("\n❌ Caso 3: Contraseña MUY CORTA:")
try:
    cuenta3 = CuentaUsuario(
        email="luis@example.com",
        contraseña="abc",  # Solo 3 caracteres
        usuario="luis"
    )
except ValidationError as e:
    print(f"   ❌ Error: {e.errors()[0]['msg']}")
    print(f"   💡 Mínimo requerido: 8 caracteres")

print("\n💡 EmailStr también puede ser None:")
class CuentaOpcional(BaseModel):
    email: Optional[EmailStr] = None  # Email opcional pero SI es email
    usuario: str

cuenta4 = CuentaOpcional(usuario="anonimo")
print(f"   {cuenta4}")

## 3. Listas y colecciones

In [ ]:
from typing import List, Optional

print("\n" + "=" * 60)
print("EJEMPLO 3: Listas y Colecciones")
print("=" * 60)

class Equipo(BaseModel):
    """
    Modelo con LISTAS tipadas.
    Pydantic valida cada elemento de la lista.
    """
    nombre: str
    miembros: List[str]              # Lista de strings
    edades: List[int]                # Lista de enteros

print("\n✅ Caso 1: Listas VÁLIDAS:")
equipo1 = Equipo(
    nombre="Equipo A",
    miembros=["Juan", "Maria", "Carlos"],
    edades=[25, 30, 28]
)
print(f"   Equipo: {equipo1}")
print(f"   Miembros: {equipo1.miembros}")
print(f"   Edades: {equipo1.edades}")

print("\n❌ Caso 2: Tipo INCORRECTO en lista:")
try:
    equipo2 = Equipo(
        nombre="Equipo B",
        miembros=["Ana", "Bob"],
        edades=[29, "treinta"]  # ERROR: "treinta" no es int
    )
except ValidationError as e:
    print(f"   ❌ Error: Elemento en lista no es del tipo correcto")
    print(f"   Error: {e.errors()[0]['msg']}")

print("\n💡 Listas opcionales (pueden estar vacías):")
class EquipoFlexible(BaseModel):
    nombre: str
    miembros: List[str] = []  # Lista vacía por defecto
    
eq = EquipoFlexible(nombre="Sin miembros")
print(f"   {eq}")

## 4. Modelos anidados

In [ ]:
print("\n" + "=" * 60)
print("EJEMPLO 4: Modelos Anidados")
print("=" * 60)

# MODELO 1: Sub-modelo (parte de otro modelo)
class Direccion(BaseModel):
    """Sub-modelo para representar una dirección."""
    calle: str
    ciudad: str
    codigoPostal: str

# MODELO 2: Modelo que CONTIENE otro modelo
class Empleado(BaseModel):
    """
    Modelo que contiene otro modelo (anidamiento).
    Pydantic valida RECURSIVAMENTE todos los niveles.
    """
    nombre: str
    puesto: str
    direccion: Direccion  # Aquí va el modelo anidado

print("\n✅ Caso 1: Crear empleado con dirección anidada:")
empleado1 = Empleado(
    nombre="Pedro",
    puesto="Developer",
    direccion={  # Diccionario que será convertido a Direccion
        "calle": "Calle Principal 123",
        "ciudad": "Buenos Aires",
        "codigoPostal": "1000"
    }
)
print(f"   {empleado1}")
print(f"   Dirección: {empleado1.direccion.ciudad}, {empleado1.direccion.calle}")

print("\n❌ Caso 2: Dirección INCOMPLETA:")
try:
    empleado2 = Empleado(
        nombre="Sofia",
        puesto="Designer",
        direccion={
            "calle": "Av. Libertador",
            # Falta 'ciudad' y 'codigoPostal'
        }
    )
except ValidationError as e:
    print(f"   ❌ Error en modelo anidado:")
    print(f"   Modelo faltante/incompleto")
    for error in e.errors():
        print(f"      - {error['loc']}: {error['msg']}")

## 5. Caso de uso: Formulario médico con validación

In [ ]:
from datetime import date

print("\n" + "=" * 60)
print("CASO REAL: Formulario Médico con Validación Avanzada")
print("=" * 60)

class Paciente(BaseModel):
    """
    Modelo de paciente para un formulario médico.
    Combina:
    - Campos obligatorios y opcionales
    - EmailStr (validación automática)
    - Listas de alergias
    - Validadores personalizados para edad y fecha
    """
    nombre: str
    edad: int
    email: EmailStr
    alergias: Optional[List[str]] = None
    fecha_cita: str
    
    @field_validator('edad')
    @classmethod
    def edad_valida_medica(cls, v):
        """Valida rango de edad para pacientes médicos."""
        if v < 0 or v > 120:
            raise ValueError('Edad debe estar entre 0 y 120 años')
        return v
    
    @field_validator('fecha_cita')
    @classmethod
    def fecha_valida(cls, v):
        """Valida que la fecha esté en formato ISO (YYYY-MM-DD)."""
        try:
            date.fromisoformat(v)  # Intenta parsear como fecha
            return v
        except ValueError:
            raise ValueError('Formato de fecha inválido. Use YYYY-MM-DD')

print("\n✅ Caso 1: Paciente VÁLIDO con todas las alergias:")
paciente1 = Paciente(
    nombre="Sofia Garcia",
    edad=45,
    email="sofia@example.com",
    alergias=["Penicilina", "Mariscos"],
    fecha_cita="2026-06-20"
)
print(f"   ✅ Paciente registrado:")
print(f"      - Nombre: {paciente1.nombre}")
print(f"      - Edad: {paciente1.edad} años")
print(f"      - Email: {paciente1.email}")
print(f"      - Alergias: {paciente1.alergias}")
print(f"      - Cita: {paciente1.fecha_cita}")

print("\n✅ Caso 2: Paciente SIN alergias (campo opcional):")
paciente2 = Paciente(
    nombre="Carlos Lopez",
    edad=50,
    email="carlos@example.com",
    fecha_cita="2026-06-21"
)
print(f"   ✅ {paciente2}")
print(f"   Alergias: {paciente2.alergias}")

print("\n❌ Caso 3: Fecha con FORMATO INCORRECTO:")
try:
    paciente3 = Paciente(
        nombre="Ana Martinez",
        edad=35,
        email="ana@example.com",
        fecha_cita="20/06/2026"  # ERROR: DD/MM/YYYY no es válido
    )
except ValidationError as e:
    print(f"   ❌ Error: {e.errors()[0]['msg']}")

print("\n❌ Caso 4: Edad FUERA de rango:")
try:
    paciente4 = Paciente(
        nombre="Pedro",
        edad=150,  # Demasiado
        email="pedro@example.com",
        fecha_cita="2026-06-22"
    )
except ValidationError as e:
    print(f"   ❌ Error: {e.errors()[0]['msg']}")

print("\n💡 Este sistema garantiza que:")
print(f"   ✅ Todos los pacientes tienen email válido")
print(f"   ✅ Las fechas siempre están en formato correcto")
print(f"   ✅ La edad está en rango médico válido")
print(f"   ✅ Podemos confiar en los datos para operaciones posteriores")

## 6. Resumen: Por qué Pydantic es crucial para LLMs

✅ **Valida tipos automáticamente** - El LLM puede salirse del script  
✅ **Parsea JSON a objetos Python** - Fácil de usar en tu código  
✅ **Genera errores claros** - Sabes exactamente qué salió mal  
✅ **Conversión bidireccional** - Puedes ir JSON ↔ Python ↔ JSON  
✅ **Validadores personalizados** - Reglas de negocio propias  

**Próximo ejercicio:** Usaremos JSON desde un LLM + Pydantic para validar la respuesta.